In [10]:
# -*- coding: utf-8 -*-
from lxml import etree
from pathlib import Path

In [11]:
pwd

'/home/hyuntae-choi/gcam-core/policy-implementation/agriculture'

In [ ]:
# -*- coding: utf-8 -*-
from lxml import etree
from pathlib import Path

INPUT_XML  = "../../input/gcamdata/xml/ag_prodchange_ref_IRR_MGMT.xml"
OUTPUT_XML = "../../input/policy/korea-2035/agriculture/smartfarm_tech.xml"
TARGET_REGION = "South Korea"
TARGET_YEARS = ["2030", "2035"]

def pick_change_tag(tech_elem):
    """
    우선순위:
    1) 이미 존재하는 태그 중 하나를 사용: agProdChange > prodChange
    2) 없으면 상위 이름에 'Ag'가 있으면 agProdChange, 아니면 prodChange
    """
    # 1) 존재 확인
    existing = tech_elem.xpath("./period/*[local-name()='agProdChange' or local-name()='prodChange']")
    if existing:
        # 같은 기술 안에서 발견된 첫 태그의 로컬명 사용
        return existing[0].tag

    # 2) 상위 경로에 'Ag' 포함 여부로 추정
    cur = tech_elem
    while cur is not None:
        lname = etree.QName(cur).localname
        if "Ag" in lname:
            return "agProdChange"
        cur = cur.getparent()

    return "prodChange"

def ensure_period_with_change(tech_elem, year, change_value="0.01"):
    # 해당 연도의 <period year="xxxx">를 찾거나 생성
    period = tech_elem.find(f"./period[@year='{year}']")
    if period is None:
        period = etree.Element("period")
        period.set("year", year)
        # GCAM XML에서 period는 보통 technology의 '직계 자식'으로 둡니다.
        tech_elem.append(period)

    # 변경 태그 결정 (agProdChange 또는 prodChange)
    tag_name = pick_change_tag(tech_elem)

    # 이미 있는 경우 값만 교체, 없으면 생성
    change_node = period.find(f"./{tag_name}")
    if change_node is None:
        change_node = etree.Element(tag_name)
        period.append(change_node)
    change_node.text = str(change_value)

def main(in_path, out_path):
    parser = etree.XMLParser(remove_blank_text=True)
    tree = etree.parse(str(in_path), parser)
    root = tree.getroot()

    # 대상 region 선택
    region = root.find(f".//region[@name='{TARGET_REGION}']")
    if region is None:
        raise ValueError(f"region name='{TARGET_REGION}' 를 찾을 수 없습니다.")

    # 모든 SupplySector 계열 탐색: local-name()에 'SupplySector'가 포함된 노드
    # (XPath 1.0엔 ends-with가 없어 contains(local-name(), 'SupplySector') 사용)
    supply_sectors = region.xpath(".//*[contains(local-name(), 'SupplySector')]")

    changed_cnt = 0
    for sec in supply_sectors:
        # 모든 Subsector
        subsectors = sec.xpath(".//*[contains(local-name(), 'Subsector')]")
        for sub in subsectors:
            # 모든 Technology
            techs = sub.xpath(".//*[contains(local-name(), 'Technology')]")
            for tech in techs:
                for y in TARGET_YEARS:
                    ensure_period_with_change(tech, y, "0.01")
                changed_cnt += 1

    # 예쁘게 출력
    xml_bytes = etree.tostring(
        root,
        pretty_print=True,
        xml_declaration=True,
        encoding="UTF-8"
    )
    Path(out_path).write_bytes(xml_bytes)
    print(f"완료: Technology {changed_cnt}개에 대해 {TARGET_YEARS}의 prod change를 0.01로 설정했습니다.")
    print(f"저장: {out_path}")

if __name__ == "__main__":
    main(INPUT_XML, OUTPUT_XML)

완료: South Korea 내 Technology 104개에 대해 2030, 2035 prod change=0.01 적용
다른 지역과 다른 연도는 삭제. 결과 저장: ../../input/policy/korea-2035/agriculture/smartfarm_tech.xml


In [15]:
# -*- coding: utf-8 -*-
from lxml import etree
from pathlib import Path

INPUT_XML  = "../../input/gcamdata/xml/ag_prodchange_ref_IRR_MGMT.xml"
OUTPUT_XML = "../../input/policy/korea-2035/agriculture/smartfarm_61.xml"
TARGET_REGION = "South Korea"
# 연도별 prod change 값 설정
TARGET_CHANGES = {
    "2030": "0.014",
    "2035": "0.019"
}

def pick_change_tag(tech_elem):
    existing = tech_elem.xpath("./period/*[local-name()='agProdChange' or local-name()='prodChange']")
    if existing:
        return existing[0].tag
    cur = tech_elem
    while cur is not None:
        lname = etree.QName(cur).localname
        if "Ag" in lname:
            return "agProdChange"
        cur = cur.getparent()
    return "prodChange"

def ensure_period_with_change(tech_elem, year, change_value):
    period = tech_elem.find(f"./period[@year='{year}']")
    if period is None:
        period = etree.Element("period")
        period.set("year", year)
        tech_elem.append(period)

    tag_name = pick_change_tag(tech_elem)
    change_node = period.find(f"./{tag_name}")
    if change_node is None:
        change_node = etree.Element(tag_name)
        period.append(change_node)
    change_node.text = str(change_value)

def main(in_path, out_path):
    parser = etree.XMLParser(remove_blank_text=True)
    tree = etree.parse(str(in_path), parser)
    root = tree.getroot()

    region = root.find(f".//region[@name='{TARGET_REGION}']")
    if region is None:
        raise ValueError(f"region name='{TARGET_REGION}' 를 찾을 수 없습니다.")

    supply_sectors = region.xpath(".//*[contains(local-name(), 'SupplySector')]")

    changed_cnt = 0
    for sec in supply_sectors:
        subsectors = sec.xpath(".//*[contains(local-name(), 'Subsector')]")
        for sub in subsectors:
            techs = sub.xpath(".//*[contains(local-name(), 'Technology')]")
            for tech in techs:
                for year, val in TARGET_CHANGES.items():
                    ensure_period_with_change(tech, year, val)
                changed_cnt += 1

    xml_bytes = etree.tostring(
        root,
        pretty_print=True,
        xml_declaration=True,
        encoding="UTF-8"
    )
    Path(out_path).write_bytes(xml_bytes)
    print(f"완료: Technology {changed_cnt}개에 대해 {list(TARGET_CHANGES.keys())} prod change 적용")
    print(f"저장: {out_path}")

if __name__ == "__main__":
    main(INPUT_XML, OUTPUT_XML)


완료: Technology 104개에 대해 ['2030', '2035'] prod change 적용
저장: ../../input/policy/korea-2035/agriculture/smartfarm_61.xml
